### Using ChromaDB for the vector database and wrapping the final response with an LLM is a classic, highly effective RAG (Retrieval-Augmented Generation) setup for customer support.

# 1. Install dependencies

### !pip install chromadb pandas anthropic sentence-transformers python-dotenv -q

# 2. Load and inspect the dataset

In [17]:
import pandas as pd

df = pd.read_csv("customer_support_tickets.csv")
df = df.dropna(subset=["Ticket Description"])  # need text to embed
print(df.shape)
df.head()

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


# 3. Build a "document" per ticket for embedding
### Combine the fields that matter for semantic search into one text blob per ticket, and keep the rest as metadata (for filtering/display).

In [18]:
def build_document(row):
    return (
        f"Ticket Subject: {row['Ticket Subject']}\n"
        f"Ticket Type: {row['Ticket Type']}\n"
        f"Description: {row['Ticket Description']}\n"
        f"Resolution: {row.get('Resolution', 'N/A')}"
    )

df["doc_text"] = df.apply(build_document, axis=1)

def build_metadata(row):
    return {
        "ticket_id": str(row["Ticket ID"]),
        "product": str(row.get("Product Purchased", "")),
        "ticket_type": str(row.get("Ticket Type", "")),
        "priority": str(row.get("Ticket Priority", "")),
        "status": str(row.get("Ticket Status", "")),
        "channel": str(row.get("Ticket Channel", "")),
    }

metadatas = df.apply(build_metadata, axis=1).tolist()
documents = df["doc_text"].tolist()
ids = [str(x) for x in df["Ticket ID"].tolist()]

# 4. Set up ChromaDB with an embedding function
### Using a local sentence-transformers model keeps this free and fast (no API calls needed just for embeddings).

In [19]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="./chroma_store")

collection = client.get_or_create_collection(
    name="support_tickets",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6738.63it/s]


# 5. Ingest data in batches (Chroma has batch size limits)

In [21]:
from tqdm import tqdm

BATCH = 500

for i in tqdm(range(0, len(documents), BATCH), desc="Adding documents"):
    collection.add(
        documents=documents[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
        ids=ids[i:i+BATCH]
    )

print("Total docs in collection:", collection.count())

Adding documents: 100%|██████████| 17/17 [00:53<00:00,  3.15s/it]

Total docs in collection: 8469


# 6. Retrieval function

In [22]:
def retrieve(query, n_results=5, where=None):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where  # e.g. {"priority": "high"}
    )
    hits = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append({"text": doc, "metadata": meta, "distance": dist})
    return hits

# 7. LLM wrapper to generate the final answer
### Using the Cerebras API — pass retrieved context + user query, get a grounded answer.

In [26]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.cerebras.ai/v1"
)

In [27]:
def generate_answer(query, hits):
    context = "\n\n---\n\n".join(
        f"[Ticket {h['metadata']['ticket_id']}] {h['text']}" for h in hits
    )

    system_prompt = (
        "You are a customer support assistant. Answer the user's question using "
        "only the ticket context provided. If the context doesn't contain a clear "
        "answer, say so honestly rather than guessing. Be concise and helpful."
    )

    user_prompt = f"Context from similar past tickets:\n{context}\n\nUser question: {query}"

    response = client.chat.completions.create(
    model="gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    temperature=1.7
)
    return response.content[0].text

# 8. RAG pipeline function tying it together

In [28]:
def rag_query(query, n_results=5, where=None, verbose=False):
    hits = retrieve(query, n_results=n_results, where=where)
    if verbose:
        print("Retrieved tickets:")
        for h in hits:
            print(f"- [{h['metadata']['ticket_id']}] dist={h['distance']:.3f}")
    answer = generate_answer(query, hits)
    return answer

# 9. CLI chatbot loop

In [29]:
def run_cli():
    print("Customer Support RAG Chatbot (type 'exit' to quit)\n")
    while True:
        query = input("You: ").strip()
        if query.lower() in ("exit", "quit"):
            print("Goodbye!")
            break
        if not query:
            continue
        try:
            answer = rag_query(query, n_results=5)
            print(f"\nBot: {answer}\n")
        except Exception as e:
            print(f"Error: {e}")

run_cli()

Customer Support RAG Chatbot (type 'exit' to quit)

Error: 'ChatCompletion' object has no attribute 'content'
Error: Error code: 429 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}
Error: 'ChatCompletion' object has no attribute 'content'
Goodbye!
